In [2]:
import os

print(f"je m'exécute depuis     : {os.getcwd()}")
print(f"le dossier data existe  : {os.path.isdir("../data/")}")
print(f"le fichier existe       : {os.path.isfile("../data/restaurants.json")}")
print(f"contenu de ../data      : {os.listdir("../data")}")

je m'exécute depuis     : c:\Users\bourr\Desktop\json-api-geocodage\notebooks
le dossier data existe  : True
le fichier existe       : True
contenu de ../data      : ['restaurants.json']


In [4]:
# 1.2 Le lire : json.load
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [5]:
# 1.3 Descendre dans l'imbrication
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [6]:
# 1.4 Parcourir
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r["nom"]:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [7]:
# 1.5 L'écrire : json.dumps
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [9]:
# 2.1 json_normalize, et la surprise
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [11]:
# 2.2 record_path : dire où sont les lignes
df = pd.json_normalize(data, record_path="restaurants")
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes  
0    glacier       [4.6, 4.8, 4.7]  
1  trattoria            [4.4, 4.5]  
2    poisson  [4.2, 4.0, 4.3, 4.1]  
3      grill            [3.9, 4.1]  


In [12]:
# 2.3 meta : remonter chercher le contexte
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [1]:
# 3.1 requests.get et le code de statut
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r= requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

200


In [2]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [5]:
# 3.5 Les en-têtes et le User-Agent
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (bourreauclemence@gmail.com)"}
r = requests.get(url, params={"q": "Catane"}, headers=headers)

print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

200
Catane 31310 Montesquieu-Volvestre


In [6]:
# 3.6 Gérer les erreurs et les délais
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur:", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

In [7]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))

True
Le serveur a répondu une erreur: 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


In [1]:
# 4.1 Une première adresse
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url=url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
reponse = r.json()

import json
print(json.dumps(reponse, ensure_ascii=False, indent=2)[:600])

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          2.308628,
          48.850699
        ]
      },
      "properties": {
        "label": "20 Avenue de Ségur 75007 Paris",
        "score": 0.9716281818181818,
        "housenumber": "20",
        "id": "75107_8909_00020",
        "banId": "c575af14-f697-464c-b70b-914cedb239a4",
        "name": "20 Avenue de Ségur",
        "postcode": "75007",
        "citycode": "75107",
        "x": 649266.35,
        "y": 6861406.23,
        "city": 


In [ ]:
# 4.2 Du GeoJSON, et un piège
f = reponse["features"][0]
lon, lat = f["geometry"]["coordinates"]
print("longitude    :", lon, " latitude :", lat)
print("label        :", f["properties"]["label"])

longitude    : 2.308628  latitude : 48.850699
label        : 20 Avenue de Ségur 75007 Paris


In [4]:
# 4.3 json_normalize sur la réponse
import pandas as pd

df = pd.json_normalize(reponse, record_path="features")
print(df.columns.tolist())

['type', 'geometry.type', 'geometry.coordinates', 'properties.label', 'properties.score', 'properties.housenumber', 'properties.id', 'properties.banId', 'properties.name', 'properties.postcode', 'properties.citycode', 'properties.x', 'properties.y', 'properties.city', 'properties.district', 'properties.context', 'properties.type', 'properties.importance', 'properties.depcode', 'properties.street', 'properties._type']


In [5]:
df["lon"] = df["geometry.coordinates"].str[0]
df["lat"] = df["geometry.coordinates"].str[1]
print(df[["properties.label", "lat", "lon"]])

                 properties.label        lat       lon
0  20 Avenue de Ségur 75007 Paris  48.850699  2.308628


In [7]:
# 4.4 Une fonction réutilisable
def geocoder_fr(adresse):
    """Renvoie (latitude, longitude) pour une adresse française, ou (None, None)."""
    url = "https://api-adresse.data.gouv.fr/search/"
    try:
        r = requests.get(url, params={"q": adresse, "limit": 1}, timeout=10)
        r.raise_for_status()
        features = r.json()["features"]
        if not features:
            return None, None
        lon, lat = features[0]["geometry"]["coordinates"]
        return lat, lon
    except requests.exceptions.RequestException:
        return None, None

In [8]:
print(geocoder_fr("20 avenue de Ségur 75007 Paris"))
print(geocoder_fr("zzzzz qwerty 99999"))

(48.850699, 2.308628)
(None, None)


In [9]:
# 5.1 Ce qui change
url = "https://nominatim.openstreetmap.org/search"
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (bourreauclemence@gmail.com)"}
params = {"q": "54 Via Pietro Mascagni, Catania, Italy", "format": "json", "limit": 1}

r = requests.get(url, params=params, headers=headers, timeout=10)
resultats = r.json()

print(resultats[0]["lat"], resultats[0]["lon"])
print(resultats[0]["display_name"])

37.5111836 15.0949140
Via Pietro Mascagni, Picanello-Ognina-Barriera-Canalicchio, Catania, Sicilia, 95131, Italia


In [10]:
import pandas as pd
df = pd.json_normalize(resultats)
print(df[["lat", "lon", "display_name"]])

          lat         lon                                       display_name
0  37.5111836  15.0949140  Via Pietro Mascagni, Picanello-Ognina-Barriera...


In [ ]:
# 5.3 La temporisation est obligatoire aussi
import time

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (bourreauclemence@gmail.com)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [12]:
import time
debut = time.time()
print(geocoder_monde("54 Via Pietro Mascagni, Catania, Italy"))
print(geocoder_monde("Piazza del Duomo, Catania, Italy"))
print(f"durée totale : {time.time() - debut:.1f} s")

(37.5111836, 15.094914)
(37.5024076, 15.0871898)
durée totale : 2.3 s


In [15]:
# 6 Afficher les points sur une carte
import folium

point = [37.5112, 15.0949]
m = folium.Map(location=point, zoom_start=14)
folium.Marker(location=point, popup="Un bon restaurant").add_to(m)
m